In [30]:
import pandas as pd
import numpy as np

file_path = 'input_data/Source File 4-19.xlsx'

sheet_names = [
    "EDFT",
    "TH Summary",
    "TH Hourly",
    "ML Store",
    "Emp List",
    "Apr Unapr OT",
    "Check Pivot",
    "Current Week OT",
    "Sheet10",
    "SCH",
    "SCH for MO"
]

dfs = {}

for sheet in sheet_names:
    dfs[sheet] = pd.read_excel(file_path, sheet_name=sheet)



d:\Program Files\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell X3100 is marked as a date but the serial value 3120000 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Program Files\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell X3521 is marked as a date but the serial value 7150000 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Program Files\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell S18505 is marked as a date but the serial value 2102692565 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Program Files\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell S23019 is marked as a date but the serial value 7856725593 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Program Files\Python311\Lib\site-packages\ope

In [31]:
import time
# Access example:
df_edft = dfs["EDFT"]
df_th_summary = dfs["TH Summary"]
df_th_hourly = dfs["TH Hourly"]
df_ml_store = dfs["ML Store"]
# df_edft.head(10)


# ----------------------------
# PREP DATA
# ----------------------------
prep_start = time.time()
df = df_th_hourly.copy()

df_th_hourly["Date"] = pd.to_datetime(df_th_hourly["Date"], errors="coerce")
df_th_hourly["Hours"] = pd.to_numeric(df_th_hourly["Hours"], errors="coerce")



In [43]:
# Join TH Hourly with ML Store on Dist Department Code = Unique ID (case-insensitive)
# Create lowercase versions for matching
df_th_hourly["Dist Dept Code Lower"] = df_th_hourly["Dist Department Code"].astype(str).str.lower()
df_ml_store["Unique ID Lower"] = df_ml_store["Unique ID"].astype(str).str.lower()

df_th_hourly_joined = df_th_hourly.merge(
    df_ml_store,
    left_on="Dist Dept Code Lower",
    right_on="Unique ID Lower",
    how="left"
)

# Drop the lowercase columns after join
df_th_hourly_joined = df_th_hourly_joined.drop(columns=["Dist Dept Code Lower", "Unique ID Lower"])

# Filter only Pay Group = ML
df_th_hourly_joined_ml = df_th_hourly_joined[df_th_hourly_joined["Pay Group"] == "ML"]

df_th_hourly_joined_ml.to_csv("output_data/th_hourly_joined_ml.csv", index=False)

print(f"Original TH Hourly rows: {len(df_th_hourly)}")
print(f"After join & filter (ML only): {len(df_th_hourly_joined_ml)}")
print(f"Matched rows: {df_th_hourly_joined_ml['Unique ID'].notna().sum()}")
print("\nJoined columns:", df_th_hourly_joined_ml.columns.tolist())

Original TH Hourly rows: 17865
After join & filter (ML only): 14556
Matched rows: 14556

Joined columns: ['EE Code', 'Key', 'Employee_Name', 'Position_Title', 'UID', 'Pay Group', 'Employee_Status', 'DOL_Status', 'Pay_Type', 'Hire/Rehire Date', 'Home Department Code', 'Home Department Desc', 'Dist Department Code', 'Dist Department Desc', 'Day', 'Date', 'Earning Code', 'Earning Desc', 'Hours', 'Unique ID', 'Market Name', 'Store', 'SD Name', 'TM Name', 'HRBP / HRG', 'Tier', 'SOM Name', 'Store Contact', 'SOL Name']


In [17]:
# # Employee daily hours report (pivot: EE Code, Employee_Name as rows, Dates as columns)
# employee_daily_pivot = df_th_hourly_joined_ml.pivot_table(
#     index=["EE Code", "Employee_Name"],
#     columns="Date",
#     values="Hours",
#     aggfunc="sum",
#     fill_value=0
# )

# # Optional: Format date columns
# # employee_daily_pivot.columns = employee_daily_pivot.columns.strftime("%d-%b-%Y")

# # Add total hours column
# employee_daily_pivot["Total Hours"] = employee_daily_pivot.sum(axis=1)

# # Save to CSV
# employee_daily_pivot.to_csv("output_data/employee_daily_hours_pivot.csv")

# # Show sample
# employee_daily_pivot.head()

,Date,2026-04-01 00:00:00,2026-04-02 00:00:00,2026-04-03 00:00:00,2026-04-04 00:00:00,2026-04-05 00:00:00,2026-04-06 00:00:00,2026-04-07 00:00:00,2026-04-08 00:00:00,2026-04-09 00:00:00,2026-04-10 00:00:00,2026-04-11 00:00:00,2026-04-12 00:00:00,2026-04-13 00:00:00,2026-04-14 00:00:00,2026-04-15 00:00:00,2026-04-16 00:00:00,2026-04-17 00:00:00,2026-04-18 00:00:00,2026-04-19 00:00:00,Total Hours
EE Code,Employee_Name,,,,,,,,,,,,,,,,,,,,
A008,"AGRAMONTE, ANNY",0.00,6.75,7.67,7.20,5.85,6.99,0.00,0.00,7.45,7.42,7.25,6.50,7.38,0.00,0.00,7.25,7.60,7.18,6.32,98.81
A00G,"AGUILAR, JAILENE",9.82,6.03,0.00,9.88,0.00,0.00,10.48,9.03,0.00,5.33,5.28,7.43,0.00,7.25,10.33,0.00,4.15,5.15,0.00,90.16
A00K,"AGUIRRE, VALERIA LIZETH",8.25,7.57,8.00,0.00,0.00,8.43,7.22,7.41,7.45,7.92,0.00,0.00,0.00,11.17,6.59,9.25,0.00,8.18,7.65,105.09
A00Z,"ALFEREZ, DAVID MANUEL",6.90,8.02,5.92,7.80,7.42,8.38,0.00,7.68,7.85,8.00,0.00,6.87,7.75,0.00,7.90,7.22,6.18,0.00,0.00,103.89
A01E,"ALVARADO, LESLY",0.00,0.00,0.00,0.00,0.00,0.00,0.00,5.97,0.00,4.95,6.95,6.70,5.50,0.00,9.58,0.00,4.23,9.22,5.57,58.67


In [45]:
import copy

# Make a DEEP copy of df_th_hourly_joined_ml
df_th_hourly_copy = copy.deepcopy(df_th_hourly_joined_ml)

# Add week column (Wednesday to Tuesday week)
# Week starts on Wednesday (weekday 2)
df_th_hourly_copy["Week_Start"] = df_th_hourly_copy["Date"] - pd.to_timedelta(
    (df_th_hourly_copy["Date"].dt.weekday - 2) % 7, unit='D'
)
df_th_hourly_copy["Week_End"] = df_th_hourly_copy["Week_Start"] + pd.Timedelta(days=6)

df_th_hourly_copy["Week_Label"] = (
    df_th_hourly_copy["Week_Start"].dt.strftime("%d-%b-%Y") +
    " to " +
    df_th_hourly_copy["Week_End"].dt.strftime("%d-%b-%Y")
)

# Identify OT rows (based on Earning Desc containing 'OT')
df_th_hourly_copy["Is_OT"] = df_th_hourly_copy["Earning Desc"].astype(str).str.contains(
    "OT", case=False, na=False
)

# Get store columns (first row per EE Code - these are constant per employee)
store_cols = ["SD Name", "TM Name", "SOM Name", "Store Contact", "SOL Name", "Store"]
employee_info = df_th_hourly_copy.groupby("Key")[store_cols].first().reset_index()

# Calculate total hours and OT hours separately
total_hours = (
    df_th_hourly_copy
    .groupby(["Key", "Week_Label"])["Hours"]
    .sum()
    .reset_index()
    .rename(columns={"Hours": "Total_Hours"})
)

ot_hours = (
    df_th_hourly_copy[df_th_hourly_copy["Is_OT"]]
    .groupby(["Key", "Week_Label"])["Hours"]
    .sum()
    .reset_index()
    .rename(columns={"Hours": "OT_Hours"})
)

# Merge
weekly_report = total_hours.merge(
    ot_hours, on=["Key", "Week_Label"], how="left"
)
weekly_report["OT_Hours"] = weekly_report["OT_Hours"].fillna(0)

# Add store info
weekly_report = weekly_report.merge(employee_info, on="Key", how="left")

# Pivot: Week labels as columns
weekly_pivot = weekly_report.pivot_table(
    index=[
        "Key", "SD Name", "TM Name", "SOM Name",
        "Store Contact", "SOL Name", "Store"
    ],
    columns="Week_Label",
    values=["Total_Hours", "OT_Hours"],
    aggfunc="sum",
    fill_value=0
)

# Flatten column names
weekly_pivot.columns = [
    f"{col[1]}_{col[0]}" for col in weekly_pivot.columns
]

# Save to CSV
weekly_pivot.to_csv("output_data/employee_weekly_ot_report_pivot.csv")

# Show sample
weekly_pivot.head()

,,,,,,,01-Apr-2026 to 07-Apr-2026_OT_Hours,08-Apr-2026 to 14-Apr-2026_OT_Hours,15-Apr-2026 to 21-Apr-2026_OT_Hours,01-Apr-2026 to 07-Apr-2026_Total_Hours,08-Apr-2026 to 14-Apr-2026_Total_Hours,15-Apr-2026 to 21-Apr-2026_Total_Hours
Key,SD Name,TM Name,SOM Name,Store Contact,SOL Name,Store,,,,,,
A00870145313,Mohammed Dadani,Eloisa Perez,Heidi Gibson,Nimia Fernandez,Heidi Gibson SOL,Pleasant Valley,0.00,0.0,0.0,34.46,36.00,28.35
A00G70144329,David Brod,Hayden Robinson,Sonia Rajani,Lisa Saldana,Zubair Alam,Uvalde,0.00,0.0,0.0,36.21,34.32,19.63
A00K70144496,Zach Wagner,Courtney Guidry,Sandra Barragan,Perla Perez Laureano,Sandra Barragan SOL,Ambassador Rd,0.00,0.0,0.0,39.47,33.95,31.67
A00Z70144874,Marc Rivera,North Texas Market - No TM,Sonia Rajani,Arturo Lugo,Sonia Rajani SOL,Kiest Blvd,4.44,0.0,0.0,44.44,38.15,21.30
A01E70144944,Oh-PA Region - No SD,Carolina - West Market - No TM,Sonia Rajani,Helen Romero,Romana Soorty,Mount Airy,0.00,0.0,0.0,0.00,0.00,5.05


In [47]:
# OT hours per week per SOM Name
ot_by_som = (
    df_th_hourly_copy[df_th_hourly_copy["Is_OT"]]
    .groupby(["SOM Name", "Week_Label"])["Hours"]
    .sum()
    .reset_index()
    .rename(columns={"Hours": "OT_Hours"})
)

# Pivot: SOM Name as rows, Week as columns
ot_by_som_pivot = ot_by_som.pivot_table(
    index="SOM Name",
    columns="Week_Label",
    values="OT_Hours",
    aggfunc="sum",
    fill_value=0
)

# Add total column
ot_by_som_pivot["Total_OT_Hours"] = ot_by_som_pivot.sum(axis=1)

# Save to CSV
ot_by_som_pivot.to_csv("output_data/ot_hours_by_som_week.csv")

# Show
ot_by_som_pivot

Week_Label,01-Apr-2026 to 07-Apr-2026,08-Apr-2026 to 14-Apr-2026,15-Apr-2026 to 21-Apr-2026,Total_OT_Hours
SOM Name,,,,
Heidi Gibson,40.75,50.14,0.00,90.89
Sandra Barragan,211.61,130.00,10.62,352.23
Sonia Rajani,287.07,252.62,29.30,568.99
Tashara League,233.71,130.82,1.40,365.93


In [15]:
# Debug: Check breakdown by Earning Desc
print("Earning Desc value counts (ML only):")
print(df_th_hourly_joined_ml["Earning Desc"].value_counts())

print("\n--- OT Hours by Earning Desc ---")
ot_df = df_th_hourly_joined_ml[df_th_hourly_joined_ml["Earning Desc"].astype(str).str.contains("OT", case=False, na=False)]
print(ot_df.groupby("Earning Desc")["Hours"].sum())

Earning Desc value counts (ML only):
Earning Desc
Regular          14083
OT                 408
Paid Time Off       56
Bereavement          7
Sick                 1
Jury Duty            1
Name: count, dtype: int64

--- OT Hours by Earning Desc ---
Earning Desc
OT    1378.04
Name: Hours, dtype: float64


In [15]:
dfs.keys()

# Access example:
df_edft = dfs["EDFT"]
df_th_summary = dfs["TH Summary"]
df_th_hourly = dfs["TH Hourly"]

# df_edft.head(10)

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows

# ----------------------------
# PREP DATA
# ----------------------------
df = df_th_hourly.copy()

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Hours"] = pd.to_numeric(df["Hours"], errors="coerce")

# ----------------------------
# PIVOT (EE Code + Name)
# ----------------------------
pivot_df = df.pivot_table(
    index=["EE Code", "Employee_Name"],
    columns="Date",
    values="Hours",
    aggfunc="sum",
    fill_value=0
)

# Sort dates
pivot_df = pivot_df.sort_index(axis=1)

# Format column names
pivot_df.columns = pivot_df.columns.strftime("%d-%b")

# Add totals
pivot_df["Total Hours"] = pivot_df.sum(axis=1)

# Reset index
pivot_df = pivot_df.reset_index()

# ----------------------------
# CREATE EXCEL
# ----------------------------
wb = Workbook()
ws = wb.active
ws.title = "Employee Hours"

# Write data
for r in dataframe_to_rows(pivot_df, index=False, header=True):
    ws.append(r)

# ----------------------------
# STYLING
# ----------------------------
header_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")

for cell in ws[1]:
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center")
    cell.fill = header_fill

# Freeze panes (lock names while scrolling dates)
ws.freeze_panes = "C2"

# Auto column width
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        try:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        except:
            pass
    ws.column_dimensions[col_letter].width = max_length + 2

# ----------------------------
# SAVE FILE
# ----------------------------
output_file = "output_data/Employee_Daily_Hours_Formatted.xlsx"
wb.save(output_file)

print("Saved:", output_file)

Saved: Employee_Daily_Hours_Formatted.xlsx


In [16]:
# ----------------------------
# STANDARDIZE COLUMNS
# ----------------------------
df_th_hourly["Date"] = pd.to_datetime(df_th_hourly["Date"], errors="coerce")
df_th_hourly["Hours"] = pd.to_numeric(df_th_hourly["Hours"], errors="coerce")

# OT IDENTIFICATION
ot_mask = df_th_hourly["Earning Desc"].str.contains("OT", case=False, na=False)

# ----------------------------
# 1. EMPLOYEE METRICS
# ----------------------------
emp_total_hours = df_th_hourly.groupby("EE Code")["Hours"].sum().rename("total_hours")

emp_ot_hours = df_th_hourly[ot_mask].groupby("EE Code")["Hours"].sum().rename("ot_hours")

emp_days_worked = df_th_hourly.groupby("EE Code")["Date"].nunique().rename("days_worked")

emp_daily_hours = df_th_hourly.groupby(["EE Code", "Date"])["Hours"].sum().reset_index()
emp_avg_daily_hours = emp_daily_hours.groupby("EE Code")["Hours"].mean().rename("avg_daily_hours")

emp_metrics = pd.concat([
    emp_total_hours,
    emp_ot_hours,
    emp_days_worked,
    emp_avg_daily_hours
], axis=1).fillna(0)

emp_metrics["regular_hours"] = emp_metrics["total_hours"] - emp_metrics["ot_hours"]
emp_metrics["ot_ratio"] = emp_metrics["ot_hours"] / emp_metrics["total_hours"].replace(0, np.nan)

# ----------------------------
# 2. POSITION METRICS
# ----------------------------
position_total_hours = df_th_hourly.groupby("Position_Title")["Hours"].sum().rename("total_hours")

position_ot_hours = df_th_hourly[ot_mask].groupby("Position_Title")["Hours"].sum().rename("ot_hours")

position_headcount = df_th_hourly.groupby("Position_Title")["EE Code"].nunique().rename("headcount")

position_metrics = pd.concat([
    position_total_hours,
    position_ot_hours,
    position_headcount
], axis=1).fillna(0)

position_metrics["avg_hours_per_employee"] = (
    position_metrics["total_hours"] / position_metrics["headcount"].replace(0, np.nan)
)

position_metrics["ot_ratio"] = (
    position_metrics["ot_hours"] / position_metrics["total_hours"].replace(0, np.nan)
)

# ----------------------------
# 3. DEPARTMENT METRICS
# ----------------------------
dept_total_hours = df_th_hourly.groupby("Dist Department Code")["Hours"].sum().rename("total_hours")

dept_ot_hours = df_th_hourly[ot_mask].groupby("Dist Department Code")["Hours"].sum().rename("ot_hours")

dept_headcount = df_th_hourly.groupby("Dist Department Code")["EE Code"].nunique().rename("headcount")

dept_metrics = pd.concat([
    dept_total_hours,
    dept_ot_hours,
    dept_headcount
], axis=1).fillna(0)

dept_metrics["avg_hours_per_employee"] = (
    dept_metrics["total_hours"] / dept_metrics["headcount"].replace(0, np.nan)
)

dept_metrics["ot_ratio"] = (
    dept_metrics["ot_hours"] / dept_metrics["total_hours"].replace(0, np.nan)
)

dept_metrics["ot_dependency"] = (
    dept_metrics["ot_hours"] /
    (dept_metrics["total_hours"] - dept_metrics["ot_hours"]).replace(0, np.nan)
)

# ----------------------------
# 4. PAY GROUP METRICS (MO vs ML)
# ----------------------------
pg_total_hours = df_th_hourly.groupby("Pay Group")["Hours"].sum().rename("total_hours")

pg_ot_hours = df_th_hourly[ot_mask].groupby("Pay Group")["Hours"].sum().rename("ot_hours")

pg_headcount = df_th_hourly.groupby("Pay Group")["EE Code"].nunique().rename("headcount")

paygroup_metrics = pd.concat([
    pg_total_hours,
    pg_ot_hours,
    pg_headcount
], axis=1).fillna(0)

paygroup_metrics["avg_hours_per_employee"] = (
    paygroup_metrics["total_hours"] / paygroup_metrics["headcount"].replace(0, np.nan)
)

paygroup_metrics["ot_ratio"] = (
    paygroup_metrics["ot_hours"] / paygroup_metrics["total_hours"].replace(0, np.nan)
)

# ----------------------------
# 5. TIME-BASED METRICS
# ----------------------------
daily_metrics = df_th_hourly.groupby("Date").agg(
    total_hours=("Hours", "sum"),
    active_employees=("EE Code", "nunique")
)

daily_metrics["avg_hours_per_employee"] = (
    daily_metrics["total_hours"] / daily_metrics["active_employees"].replace(0, np.nan)
)

daily_ot = df_th_hourly[ot_mask].groupby("Date")["Hours"].sum().rename("ot_hours")

daily_metrics = daily_metrics.join(daily_ot).fillna(0)

# WEEKLY
df_th_hourly["Week"] = df_th_hourly["Date"].dt.isocalendar().week

weekly_metrics = df_th_hourly.groupby("Week").agg(
    total_hours=("Hours", "sum"),
    active_employees=("EE Code", "nunique")
)

weekly_ot = df_th_hourly[ot_mask].groupby(df_th_hourly["Date"].dt.isocalendar().week)["Hours"].sum().rename("ot_hours")

weekly_metrics = weekly_metrics.join(weekly_ot).fillna(0)

weekly_metrics["wow_growth"] = weekly_metrics["total_hours"].pct_change()

# DAY OF WEEK
df_th_hourly["Day_Name"] = df_th_hourly["Date"].dt.day_name()

dow_metrics = df_th_hourly.groupby("Day_Name")["Hours"].mean().rename("avg_hours")

# ----------------------------
# 6. EARNING CODE METRICS
# ----------------------------
earning_metrics = df_th_hourly.groupby(["Earning Code", "Earning Desc"])["Hours"].sum().rename("total_hours").reset_index()

# ----------------------------
# 7. UTILIZATION METRICS
# ----------------------------
active_emp_daily = df_th_hourly.groupby("Date")["EE Code"].nunique()
total_hours_daily = df_th_hourly.groupby("Date")["Hours"].sum()

utilization_metrics = (total_hours_daily / active_emp_daily).rename("hours_per_employee")

# ----------------------------
# 8. CROSS DIMENSIONAL METRICS
# ----------------------------
# Employee × Department
emp_dept_matrix = df_th_hourly.groupby(["EE Code", "Dist Department Code"])["Hours"].sum().reset_index()

# Position × Department
pos_dept_matrix = df_th_hourly.groupby(["Position_Title", "Dist Department Code"])["Hours"].sum().reset_index()

# PayGroup × Department
pg_dept_matrix = df_th_hourly.groupby(["Pay Group", "Dist Department Code"])["Hours"].sum().reset_index()

In [17]:
import streamlit as st

# ----------------------------
# DAILY TREND
# ----------------------------
st.subheader("📈 Daily Hours Trend")

daily = df_filtered.groupby("Date")["Hours"].sum().reset_index()
fig_daily = px.line(daily, x="Date", y="Hours", title="Daily Total Hours")
st.plotly_chart(fig_daily, use_container_width=True)

# ----------------------------
# WEEKLY TREND
# ----------------------------
st.subheader("📊 Weekly Trend")

df_filtered["Week"] = df_filtered["Date"].dt.isocalendar().week
weekly = df_filtered.groupby("Week")["Hours"].sum().reset_index()
fig_weekly = px.bar(weekly, x="Week", y="Hours", title="Weekly Hours")
st.plotly_chart(fig_weekly, use_container_width=True)

# ----------------------------
# POSITION ANALYSIS
# ----------------------------
st.subheader("👔 Position Analysis")

pos = df_filtered.groupby("Position_Title")["Hours"].sum().reset_index().sort_values(by="Hours", ascending=False)
fig_pos = px.bar(pos, x="Hours", y="Position_Title", orientation="h", title="Hours by Position")
st.plotly_chart(fig_pos, use_container_width=True)

# ----------------------------
# DEPARTMENT ANALYSIS
# ----------------------------
st.subheader("🏢 Department Analysis")

dept = df_filtered.groupby("Dist Department Code")["Hours"].sum().reset_index()
fig_dept = px.pie(dept, names="Dist Department Code", values="Hours", title="Department Share")
st.plotly_chart(fig_dept, use_container_width=True)

# ----------------------------
# OT ANALYSIS
# ----------------------------
st.subheader("⏱️ Overtime Analysis")

ot_df = df_filtered[df_filtered["Earning Desc"].str.contains("OT", case=False, na=False)]
ot_by_emp = ot_df.groupby("EE Code")["Hours"].sum().reset_index().sort_values(by="Hours", ascending=False).head(10)

fig_ot = px.bar(ot_by_emp, x="EE Code", y="Hours", title="Top 10 Employees by OT")
st.plotly_chart(fig_ot, use_container_width=True)

# ----------------------------
# DAY OF WEEK ANALYSIS
# ----------------------------
st.subheader("📅 Day of Week Analysis")

df_filtered["Day_Name"] = df_filtered["Date"].dt.day_name()
dow = df_filtered.groupby("Day_Name")["Hours"].mean().reset_index()

fig_dow = px.bar(dow, x="Day_Name", y="Hours", title="Average Hours by Day")
st.plotly_chart(fig_dow, use_container_width=True)

# ----------------------------
# UTILIZATION
# ----------------------------
st.subheader("⚙️ Utilization")

util = df_filtered.groupby("Date").agg(total_hours=("Hours", "sum"), employees=("EE Code", "nunique"))
util["hours_per_employee"] = util["total_hours"] / util["employees"]
util = util.reset_index()

fig_util = px.line(util, x="Date", y="hours_per_employee", title="Hours per Employee")
st.plotly_chart(fig_util, use_container_width=True)

# ----------------------------
# RAW DATA VIEW
# ----------------------------
st.subheader("📄 Raw Data")
st.dataframe(df_filtered.head(100))


2026-04-21 03:34:07.125 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-21 03:34:07.460 
  command:

    streamlit run C:\Users\Tested\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-21 03:34:07.462 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-21 03:34:07.463 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name 'df_filtered' is not defined